# Embeddings y procesamiento de texto para LLMs

Este notebook sigue el flujo del Capitulo 2 de *Build a Large Language Model (From Scratch)* de Sebastian Raschka, adaptando el codigo y explicaciones en espanol y en mi propio estilo. Aqui se exploran los pasos clave para procesar texto y generar embeddings, fundamentales para modelos de lenguaje y sistemas agenticos.

**Autor:** Diego Forero | 2026

## 1. Importar librerias necesarias

En este paso importamos las librerias fundamentales para el procesamiento de texto y la construccion de embeddings. Usaremos `torch` para las redes neuronales y `tiktoken` para la tokenizacion eficiente, ademas de utilidades estandar de Python.

Importar correctamente estas librerias es esencial para aprovechar las capacidades modernas de procesamiento y modelado de lenguaje.

In [1]:
import torch
import tiktoken
import re
import os
import requests
from torch.utils.data import Dataset, DataLoader

print('torch:', torch.__version__)
print('tiktoken:', tiktoken.__version__)

torch: 2.10.0+cpu
tiktoken: 0.12.0


## 2. Descargar y cargar el texto

En este paso cargamos el archivo `the-verdict.txt`, que contiene el texto base para el procesamiento. Tener el texto localmente permite reproducibilidad y control sobre los datos de entrada.

In [2]:
file_path = 'the-verdict.txt'
if not os.path.exists(file_path):
    url = 'https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt'
    response = requests.get(url)
    response.raise_for_status()
    with open(file_path, 'wb') as f:
        f.write(response.content)

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print('Total de caracteres:', len(raw_text))
print(raw_text[:200])

Total de caracteres: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a


## 3. Tokenizacion del texto

Utilizamos `tiktoken` para convertir el texto en una secuencia de tokens numericos. Esta representacion es la base para que el modelo pueda trabajar con el texto.

La tokenizacion eficiente es clave para que el modelo entienda la estructura y el significado del texto.

In [3]:
tokenizer = tiktoken.get_encoding('gpt2')
tokens = tokenizer.encode(raw_text, allowed_special={'<|endoftext|>'})
print('Total de tokens:', len(tokens))
print('Primeros 20 tokens:', tokens[:20])

Total de tokens: 5145
Primeros 20 tokens: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


### Tokenizacion manual paso a paso (del libro)

Antes de usar BPE, el libro muestra como construir un tokenizador sencillo desde cero usando regex. Esto ayuda a entender por que BPE es superior.

In [4]:
# Tokenizacion con regex (SimpleTokenizerV1)
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print('Tokens con regex:', len(preprocessed))
print(preprocessed[:30])

# Construir vocabulario
all_words = sorted(set(preprocessed))
print('Tamano del vocabulario:', len(all_words))
vocab = {token: idx for idx, token in enumerate(all_words)}

Tokens con regex: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
Tamano del vocabulario: 1130


In [5]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer_v1 = SimpleTokenizerV1(vocab)
sample = '"It\'s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'
ids = tokenizer_v1.encode(sample)
print('Encoded:', ids)
print('Decoded:', tokenizer_v1.decode(ids))

Encoded: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
Decoded: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


### SimpleTokenizerV2 - Manejando palabras desconocidas

In [6]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(['<|endoftext|>', '<|unk|>'])
vocab = {token: idx for idx, token in enumerate(all_tokens)}

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int else '<|unk|>' for item in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        text = ' '.join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

tokenizer_v2 = SimpleTokenizerV2(vocab)
text1 = 'Hello, do you like tea?'
text2 = 'In the sunlit terraces of the palace.'
text = ' <|endoftext|> '.join((text1, text2))
print(tokenizer_v2.decode(tokenizer_v2.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 4. Division del texto en fragmentos (chunks) con sliding window

Dividimos la secuencia de tokens en fragmentos de longitud fija usando `max_length` y `stride`. El solapamiento (overlap) entre fragmentos permite que el modelo tenga acceso a contexto compartido entre diferentes partes del texto.

Los LLMs se entrenan prediciendo el *siguiente* token dada una secuencia de tokens anteriores. La ventana deslizante genera exactamente estos pares: una secuencia de entrada (X) y una secuencia objetivo (Y) desplazada una posicion.

In [7]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

# Probamos con batch_size=8, max_length=4, stride=4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print('Inputs:\n', inputs)
print('\nTargets:\n', targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


In [8]:
# Funcion simple para ver fragmentos
def chunk_tokens(tokens, max_length=128, stride=64):
    chunks = []
    for i in range(0, len(tokens) - max_length + 1, stride):
        chunk = tokens[i:i+max_length]
        chunks.append(chunk)
    return chunks

tokenizer_bpe = tiktoken.get_encoding('gpt2')
tokens = tokenizer_bpe.encode(raw_text)
max_length = 128
stride = 64
chunks = chunk_tokens(tokens, max_length, stride)
print(f'Fragmentos generados: {len(chunks)}')
print('Primer fragmento (primeros 20 tokens):', chunks[0][:20])

Fragmentos generados: 79
Primer fragmento (primeros 20 tokens): [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 922, 5891, 1576, 438]


## 5. Construccion de embeddings

Para cada fragmento, generamos embeddings usando una capa de embedding de `torch`. Esto transforma los IDs de tokens en vectores densos que capturan relaciones semanticas.

Ademas, GPT-2 usa **embeddings posicionales**: otra capa de embedding que le dice al modelo en que posicion esta cada token dentro de la secuencia. La entrada final al modelo es la **suma** de los embeddings de token + los embeddings de posicion.

In [9]:
vocab_size = 50257
embedding_dim = 256
max_length = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_dim)
pos_embedding_layer = torch.nn.Embedding(max_length, embedding_dim)

# Usamos los inputs del dataloader anterior
token_embeddings = token_embedding_layer(inputs)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))

input_embeddings = token_embeddings + pos_embeddings
print('Forma de los embeddings de entrada:', input_embeddings.shape)

Forma de los embeddings de entrada:

 torch.Size([8, 4, 256])


In [10]:
# Ejemplo simple con un solo fragmento
tensor_chunk = torch.tensor(chunks[0], dtype=torch.long)
embedding_layer = torch.nn.Embedding(tokenizer_bpe.n_vocab, 64)
embeddings = embedding_layer(tensor_chunk)
print('Forma de los embeddings de un fragmento:', embeddings.shape)

Forma de los embeddings de un fragmento: torch.Size([128, 64])


## 6. Por que los embeddings codifican significado y como se relacionan con las redes neuronales?

Los embeddings transforman tokens en vectores densos que capturan relaciones semanticas. Esto es posible porque la red neuronal aprende a asignar vectores similares a palabras con significados relacionados durante el entrenamiento.

Vamos por partes:

- Un ID de token es solo un numero arbitrario: que "gato" tenga ID 42 y "perro" ID 789 no dice nada sobre su relacion.
- Un embedding transforma ese ID en un **vector de numeros reales** en un espacio de alta dimension.
- Durante el entrenamiento, el modelo aprende a **posicionar** estos vectores de tal forma que palabras que aparecen en contextos similares quedan **geometricamente cerca**.

Desde el punto de vista de redes neuronales, `torch.nn.Embedding` es una **tabla de pesos** (una matriz). Matematicamente es equivalente a hacer un one-hot encoding y multiplicarlo por una capa fully-connected. La diferencia es que el Embedding lo hace como un lookup eficiente.

Estos pesos se **actualizan por backpropagation** durante el entrenamiento. El modelo mueve los vectores para minimizar el error de prediccion, y como resultado, tokens que comparten contexto terminan agrupados. Asi los embeddings permiten que el modelo entienda y procese el significado del texto de manera eficiente.

## 7. Experimento: Cambiar max_length y stride, analizar resultados

Vamos a modificar los valores de `max_length` y `stride` para ver como afecta la cantidad de fragmentos generados y el solapamiento entre ellos.

In [11]:
tokenizer_exp = tiktoken.get_encoding('gpt2')
total_tokens = len(tokenizer_exp.encode(raw_text))
print(f'Total de tokens en el corpus: {total_tokens}\n')

# Tabla de resultados
print(f'| {"max_length":<10} | {"stride":<10} | {"Muestras":<10} |')
print('|------------|------------|------------|')

configs = [
    (4, 1), (4, 2), (4, 4),
    (16, 1), (16, 8), (16, 16),
    (64, 1), (64, 32), (64, 64),
    (128, 32), (128, 64), (128, 128),
    (256, 1), (256, 128), (256, 256)
]

for ml, st in configs:
    ds = GPTDatasetV1(raw_text, tokenizer_exp, max_length=ml, stride=st)
    print(f'| {ml:<10} | {st:<10} | {len(ds):<10} |')

Total de tokens en el corpus: 5145

| max_length | stride     | Muestras   |
|------------|------------|------------|


| 4          | 1          | 5141       |
| 4          | 2          | 2571       |


| 4          | 4          | 1286       |


| 16         | 1          | 5129       |
| 16         | 8          | 642        |
| 16         | 16         | 321        |


| 64         | 1          | 5081       |
| 64         | 32         | 159        |
| 64         | 64         | 80         |
| 128        | 32         | 157        |
| 128        | 64         | 79         |
| 128        | 128        | 40         |


| 256        | 1          | 4889       |
| 256        | 128        | 39         |
| 256        | 256        | 20         |


In [12]:
# Ejemplo visual de solapamiento
def experimenta_chunks(tokens, max_length, stride):
    chunks = chunk_tokens(tokens, max_length, stride)
    print(f"max_length={max_length}, stride={stride} => fragmentos: {len(chunks)}")
    return chunks

# Caso 1: sin solapamiento
chunks1 = experimenta_chunks(tokens, max_length=128, stride=128)
# Caso 2: mucho solapamiento
chunks2 = experimenta_chunks(tokens, max_length=128, stride=32)

print('\nEjemplo de solapamiento (primeros 10 tokens de los primeros 2 fragmentos, caso 2):')
print('Fragmento 1:', chunks2[0][:10])
print('Fragmento 2:', chunks2[1][:10])

max_length=128, stride=128 => fragmentos: 40
max_length=128, stride=32 => fragmentos: 157

Ejemplo de solapamiento (primeros 10 tokens de los primeros 2 fragmentos, caso 2):
Fragmento 1: [40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
Fragmento 2: [287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710]


### Analisis del experimento

Un stride pequeno genera mayor solapamiento: los mismos tokens aparecen en multiples ventanas y en distintas posiciones relativas, dando al modelo mas oportunidades de aprender dependencias de largo alcance. Con `stride == max_length` el solapamiento es cero — los tokens en el borde de una ventana nunca actuan como contexto para la siguiente, creando puntos ciegos. Un stride equivalente al 50-75% del `max_length` es el balance estandar entre cobertura contextual y eficiencia computacional.

## 8. Preprocesamiento y tokenizacion

El modelo no opera sobre texto crudo sino sobre secuencias de indices numericos. La tokenizacion define el vocabulario del modelo y determina como se segmenta el texto en esas unidades. Una tokenizacion deficiente fragmenta palabras de forma que oscurece patrones morfologicos y semanticos, degradando la calidad del aprendizaje desde el paso inicial.

## 9. Utilidad del solapamiento (overlap) en fragmentos

Cuando un texto se divide en fragmentos de longitud fija, la informacion que cruza el limite entre dos fragmentos queda partida y sin contexto en ambos lados. El overlap replica intencionalmente esa zona de frontera en los dos fragmentos adyacentes, garantizando que ningun segmento de texto sea procesado sin su contexto inmediato.

## 10. Rol de los embeddings en sistemas agenticos y LLMs

Los embeddings son representaciones vectoriales que traducen texto en coordenadas numericas dentro de un espacio de alta dimension. La posicion de cada vector no es arbitraria — elementos con significado similar quedan geometricamente cercanos. Esto permite que los agentes operen sobre significado real: en lugar de comparar caracteres, comparan distancias vectoriales. Es lo que habilita la busqueda semantica en bases de datos vectoriales y es la representacion interna sobre la que el modelo razona y genera texto.